<h3><b>Implement Gradient Checking</b></h3>

In [3]:
import numpy as np

In [4]:
def gradient_check_n_test_case(): 
    np.random.seed(1)
    x = np.random.randn(4,3)
    y = np.array([1, 1, 0])
    W1 = np.random.randn(5,4) 
    b1 = np.random.randn(5,1) 
    W2 = np.random.randn(3,5) 
    b2 = np.random.randn(3,1) 
    W3 = np.random.randn(1,3) 
    b3 = np.random.randn(1,1) 
    parameters = {"W1": W1,
                  "b1": b1,
                  "W2": W2,
                  "b2": b2,
                  "W3": W3,
                  "b3": b3}

    
    return x, y, parameters

In [5]:
def dictionary_to_vector(parameters):
    """
    Roll all our parameters dictionary into a single vector satisfying our specific required shape.
    """
    keys = []
    count = 0
    params = list(parameters.keys())
    
    for key in params: 

        new_vector = np.reshape(parameters[key], (-1, 1))
        keys.append((key, new_vector.shape[0], parameters[key].shape))
        
        if count == 0:
            theta = new_vector
        else:
            theta = np.concatenate((theta, new_vector), axis=0)
        count = count + 1

    return theta, keys

In [6]:
X, Y, params = gradient_check_n_test_case()

t, k = dictionary_to_vector(params)
print(t)
print(k)
print(t.shape)

[[-0.3224172 ]
 [-0.38405435]
 [ 1.13376944]
 [-1.09989127]
 [-0.17242821]
 [-0.87785842]
 [ 0.04221375]
 [ 0.58281521]
 [-1.10061918]
 [ 1.14472371]
 [ 0.90159072]
 [ 0.50249434]
 [ 0.90085595]
 [-0.68372786]
 [-0.12289023]
 [-0.93576943]
 [-0.26788808]
 [ 0.53035547]
 [-0.69166075]
 [-0.39675353]
 [-0.6871727 ]
 [-0.84520564]
 [-0.67124613]
 [-0.0126646 ]
 [-1.11731035]
 [ 0.2344157 ]
 [ 1.65980218]
 [ 0.74204416]
 [-0.19183555]
 [-0.88762896]
 [-0.74715829]
 [ 1.6924546 ]
 [ 0.05080775]
 [-0.63699565]
 [ 0.19091548]
 [ 2.10025514]
 [ 0.12015895]
 [ 0.61720311]
 [ 0.30017032]
 [-0.35224985]
 [-1.1425182 ]
 [-0.34934272]
 [-0.20889423]
 [ 0.58662319]
 [ 0.83898341]
 [ 0.93110208]
 [ 0.28558733]]
[('W1', 20, (5, 4)), ('b1', 5, (5, 1)), ('W2', 15, (3, 5)), ('b2', 3, (3, 1)), ('W3', 3, (1, 3)), ('b3', 1, (1, 1))]
(47, 1)


In [7]:
def vector_to_dictionary(theta, keys):
    """
    Unroll all our parameters dictionary from a single vector satisfying our specific required shape.
    """
    parameters = {}
    count = 0
    for key in keys:
        
        parameters[key[0]] = theta[count: count + key[1]].reshape(key[-1])
        count += key[1]
        
    return parameters

In [13]:
k = [('W1', 10, (5, 2)), ('b1', 5, (5, 1))]
t = np.array([1,2,3,4,5,6,7,8,9, 10, 11, 12, 13, 14, 15]).reshape((15, 1))

p = vector_to_dictionary(t, k)
p


{'W1': array([[ 1,  2],
        [ 3,  4],
        [ 5,  6],
        [ 7,  8],
        [ 9, 10]]),
 'b1': array([[11],
        [12],
        [13],
        [14],
        [15]])}

In [34]:
def gradients_to_vector(gradients, keys):
    """
    Roll all our gradients dictionary into a single vector satisfying our specific required shape.
    """
    new_keys = []
    for key in keys:
        new_keys.append("d" + key[0])
        
    count = 0
    theta = None
    
    for key in new_keys: 

        new_vector = np.reshape(gradients[key], (-1, 1))

        if count == 0:
            theta = new_vector
        else:
            theta = np.concatenate((theta, new_vector), axis=0)
        count = count + 1

    return theta

In [24]:

def sigmoid(Z):

        A = 1 / (1 + np.exp(-Z))
       

        return A

def relu(Z):

    A = np.maximum(0, Z)

    return A


def forward_propagation(X, Y, parameters):
  
    m = X.shape[1]
    W1 = parameters["W1"]
    b1 = parameters["b1"]
    W2 = parameters["W2"]
    b2 = parameters["b2"]
    W3 = parameters["W3"]
    b3 = parameters["b3"]

    
    Z1 = np.dot(W1, X) + b1
    A1 = relu(Z1)
    Z2 = np.dot(W2, A1) + b2
    A2 = relu(Z2)
    Z3 = np.dot(W3, A2) + b3
    A3 = sigmoid(Z3)

    
    log_probs = np.multiply(-np.log(A3),Y) + np.multiply(-np.log(1 - A3), 1 - Y)
    cost = 1. / m * np.sum(log_probs)
    
    cache = (Z1, A1, W1, b1, Z2, A2, W2, b2, Z3, A3, W3, b3)
    
    return cost, cache

In [41]:
def backward_propagation(X, Y, cache):
  
    m = X.shape[1]
    (Z1, A1, W1, b1, Z2, A2, W2, b2, Z3, A3, W3, b3) = cache
    
    dZ3 = A3 - Y
    dW3 = 1. / m * np.dot(dZ3, A2.T)
    db3 = 1. / m * np.sum(dZ3, axis=1, keepdims=True)
    
    dA2 = np.dot(W3.T, dZ3)
    dZ2 = np.multiply(dA2, np.int64(A2 > 0)) # * 2#  # correction in this line:  (* 2)  at the end of equation was deleted.
    dW2 = 1. / m * np.dot(dZ2, A1.T) 
    db2 = 1. / m * np.sum(dZ2, axis=1, keepdims=True)
    
    dA1 = np.dot(W2.T, dZ2)
    dZ1 = np.multiply(dA1, np.int64(A1 > 0))
    dW1 = 1. / m * np.dot(dZ1, X.T)
    db1 = 1. / m * np.sum(dZ1, axis=1, keepdims=True) # correction in this line:  (4. / m)  -> (1. / m)
    
    gradients = {"dZ3": dZ3, "dW3": dW3, "db3": db3,
                 "dA2": dA2, "dZ2": dZ2, "dW2": dW2, "db2": db2,
                 "dA1": dA1, "dZ1": dZ1, "dW1": dW1, "db1": db1}
    
    return gradients

In [26]:
def gradient_check(parameters, gradients, X, Y, epsilon=1e-7, trigger=2e-7, print_msg=False):
  
    parameters_values, keys = dictionary_to_vector(parameters)
    
    grad = gradients_to_vector(gradients, keys)
    num_parameters = parameters_values.shape[0]
    J_plus = np.zeros((num_parameters, 1))
    J_minus = np.zeros((num_parameters, 1))
    gradapprox = np.zeros((num_parameters, 1))
    
    
    for i in range(num_parameters):
        
        theta_plus = np.copy(parameters_values)
        theta_plus[i] = theta_plus[i] + epsilon
        J_plus[i], _ = forward_propagation(X, Y, vector_to_dictionary(theta_plus, keys))
        
     
        theta_minus = np.copy(parameters_values)
        theta_minus[i] = theta_minus[i] - epsilon
        J_minus[i], _ = forward_propagation(X, Y, vector_to_dictionary(theta_minus, keys))
          
       
        gradapprox[i] = (J_plus[i] - J_minus[i]) / (2 * epsilon)
     
    numerator = np.linalg.norm(gradapprox - grad)
    denominator = np.linalg.norm(gradapprox) + np.linalg.norm(grad)
    difference = numerator / denominator
    
    
    if print_msg:
        if difference > trigger:
            print ("\033[93m" + "There is a mistake in the backward propagation! difference = " + str(difference) + "\033[0m")
        else:
            print ("\033[92m" + "Your backward propagation works perfectly fine! difference = " + str(difference) + "\033[0m")

    return difference

In [43]:
X, Y, params = gradient_check_n_test_case()
cost, cache = forward_propagation(X, Y, params)
gradients = backward_propagation(X, Y, cache)

In [38]:
differences = gradient_check(parameters=params, gradients=gradients, X=X, Y=Y, epsilon=1e-7, trigger=2e-7, print_msg=True)

There is a mistake in the backward propagation! difference = 0.4731196947521996


In [44]:
# after corrections
differences = gradient_check(parameters=params, gradients=gradients, X=X, Y=Y, epsilon=1e-7, trigger=2e-7, print_msg=True)

Your backward propagation works perfectly fine! difference = 1.1890913024229996e-07
